# MTikGuard cross-modal fairness audit

Runs the pipeline end to end on Kaggle: text extraction (Whisper, EasyOCR), fine-tuning, and the subgroup audit.

Requires a GPU (T4 x2), internet access, and two inputs: the `anhoangvo/tikharm-dataset` dataset and this repository's `code/` folder uploaded as a dataset.

## Setup

In [ ]:
import subprocess

def find_path(pattern, kind="f"):
    result = subprocess.run(
        ["find", "/kaggle/input", "-iname", pattern, "-type", kind],
        capture_output=True, text=True,
    )
    paths = [p for p in result.stdout.strip().split("\n") if p]
    if not paths:
        raise FileNotFoundError(
            f"no match for {pattern!r} under /kaggle/input"
        )
    return paths[0]

CODE_DIR = find_path("config.py").rsplit("/", 1)[0]
print("CODE_DIR:", CODE_DIR)

TIKHARM_DIR = find_path("Dataset", kind="d")
print("TIKHARM_DIR:", TIKHARM_DIR)

In [ ]:
!mkdir -p /kaggle/working/code
!cp -r {CODE_DIR}/* /kaggle/working/code/
%cd /kaggle/working/code
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())

In [ ]:
def run(args):
    print("$", " ".join(args))
    subprocess.run(args, check=True)

## Smoke test

A few videos per class, to check the setup in under a minute.

In [ ]:
run(["python", "preprocess.py", "--smoke-test",
     "--data-dir", TIKHARM_DIR,
     "--output", "/kaggle/working/data/processed/smoke.csv"])

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/data/processed/smoke.csv")
print(df["language"].value_counts())
print(df["class_name"].value_counts())
df[["video_name", "class_name", "language", "combined_text"]].sample(5)

In [ ]:
run(["python", "train.py", "--smoke-test",
     "--csv", "/kaggle/working/data/processed/smoke.csv",
     "--output-dir", "/kaggle/working/outputs/smoke_checkpoints"])

In [ ]:
run(["python", "audit.py",
     "--checkpoint", "/kaggle/working/outputs/smoke_checkpoints/final",
     "--csv", "/kaggle/working/data/processed/smoke.csv",
     "--output", "/kaggle/working/outputs/smoke_audit.json"])

## Full run

Preprocessing resumes from the existing output CSV if it is interrupted (use `--overwrite` to start again).

In [ ]:
run(["python", "preprocess.py",
     "--data-dir", TIKHARM_DIR,
     "--output", "/kaggle/working/data/processed/tikharm_vi_en_complete.csv"])

In [ ]:
run(["python", "train.py",
     "--csv", "/kaggle/working/data/processed/tikharm_vi_en_complete.csv",
     "--output-dir", "/kaggle/working/outputs/checkpoints"])

In [ ]:
run(["python", "audit.py",
     "--checkpoint", "/kaggle/working/outputs/checkpoints/final",
     "--csv", "/kaggle/working/data/processed/tikharm_vi_en_complete.csv",
     "--output", "/kaggle/working/outputs/audit_results.json"])

## Results

In [ ]:
import json
with open("/kaggle/working/outputs/audit_results.json") as f:
    print(json.dumps(json.load(f), indent=2))